# 01 — Exploratory Data Analysis (EDA)
**Project:** Casting Defect Detection
**Dataset:** [Real-life Industrial Dataset of Casting Product](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product) (Kaggle)

**Goal of this notebook:** understand the data *before* building any model.

1. How many images are there per class and per split (train/test)?
2. Is the dataset balanced?
3. What do OK vs. defective products look like?
4. Are image size and color format consistent?
5. Do pixel intensities differ between classes?
6. Are there duplicate images between train and test (data leakage)?

> Run on **Google Colab** (Runtime → Run all). No GPU is needed for this notebook.

## 1. Setup

In [ ]:
import os
import hashlib
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)
np.random.seed(42)

# Folder where all figures from this notebook are saved (for the README / report)
FIG_DIR = "reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

## 2. Download the dataset
`kagglehub` downloads public Kaggle datasets directly. If Colab asks for Kaggle credentials,
log in with your Kaggle account (Kaggle → Settings → *Create New Token*).

In [ ]:
!pip install -q kagglehub
import kagglehub

DATA_ROOT = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")
print("Downloaded to:", DATA_ROOT)

## 3. Locate the train / test folders
The dataset contains a folder with `train/` and `test/`, each holding `def_front/` (defective)
and `ok_front/` (good). Instead of hard-coding the path, we search for it.

In [ ]:
def find_split_root(root):
    """Return the folder that contains both 'train' and 'test' with class subfolders."""
    for dirpath, dirnames, _ in os.walk(root):
        if "train" in dirnames and "test" in dirnames:
            train_dir = os.path.join(dirpath, "train")
            if {"def_front", "ok_front"} <= set(os.listdir(train_dir)):
                return dirpath
    raise FileNotFoundError("Could not find train/test folders with def_front and ok_front")

SPLIT_ROOT = find_split_root(DATA_ROOT)
print("Split root:", SPLIT_ROOT)
for split in ["train", "test"]:
    print(split, "->", sorted(os.listdir(os.path.join(SPLIT_ROOT, split))))

## 4. Build a table of all images

In [ ]:
IMG_EXT = (".jpeg", ".jpg", ".png")
LABEL_NAMES = {"def_front": "defective", "ok_front": "ok"}

rows = []
for split in ["train", "test"]:
    for cls in ["def_front", "ok_front"]:
        folder = os.path.join(SPLIT_ROOT, split, cls)
        for fname in os.listdir(folder):
            if fname.lower().endswith(IMG_EXT):
                rows.append({
                    "split": split,
                    "class": LABEL_NAMES[cls],
                    "path": os.path.join(folder, fname),
                })

df = pd.DataFrame(rows)
print("Total images:", len(df))
df.head()

## 5. Class distribution

In [ ]:
counts = df.groupby(["split", "class"]).size().unstack(fill_value=0)
counts["total"] = counts.sum(axis=1)
counts["defective_%"] = (counts["defective"] / counts["total"] * 100).round(1)
display(counts)

ratio = counts.loc["train", "defective"] / counts.loc["train", "ok"]
print(f"Train defective : ok ratio = {ratio:.2f} : 1")

In [ ]:
ax = counts[["defective", "ok"]].plot(kind="bar", figsize=(7, 4), color=["#d62728", "#2ca02c"], rot=0)
ax.set_title("Number of images per split and class")
ax.set_xlabel("Split")
ax.set_ylabel("Number of images")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_class_distribution.png", dpi=150)
plt.show()

**Notes to fill in after running:**
- Is the dataset balanced? *(e.g. defective is ~X% of train)*
- Implication for modeling: a mild imbalance means **accuracy alone can be misleading** →
  we will track recall on the defective class and may use class weights.

## 6. Sample images: OK vs. defective

In [ ]:
def show_samples(cls, n=8, split="train"):
    paths = df[(df["split"] == split) & (df["class"] == cls)]["path"].sample(n, random_state=42).tolist()
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.6))
    for ax, p in zip(axes, paths):
        ax.imshow(Image.open(p), cmap="gray")
        ax.axis("off")
    fig.suptitle(f"{cls.upper()} samples ({split})", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/02_samples_{cls}.png", dpi=150)
    plt.show()

show_samples("ok")
show_samples("defective")

**Notes to fill in after running:** describe what defects look like
*(e.g. holes/porosity on the surface, rough edges, missing material)*.

## 7. Image size and color format

In [ ]:
sample = df.sample(500, random_state=42)
sizes = Counter()
modes = Counter()
for p in sample["path"]:
    with Image.open(p) as im:
        sizes[im.size] += 1
        modes[im.mode] += 1

print("Image sizes (width, height):", dict(sizes))
print("Color modes:", dict(modes))

**Why this matters:** pretrained models (MobileNetV2, EfficientNet, ResNet) expect
**3-channel RGB** input of a fixed size (e.g. 224×224). If the images are grayscale or a
different size, we must convert/resize them in preprocessing.

## 8. Pixel intensity per class

In [ ]:
def mean_intensity(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.float32).mean()

train_sample = df[df["split"] == "train"].groupby("class", group_keys=False).sample(400, random_state=42)
train_sample["mean_intensity"] = train_sample["path"].apply(mean_intensity)

plt.figure(figsize=(7, 4))
for cls, color in [("ok", "#2ca02c"), ("defective", "#d62728")]:
    vals = train_sample[train_sample["class"] == cls]["mean_intensity"]
    plt.hist(vals, bins=30, alpha=0.6, label=cls, color=color)
plt.title("Mean pixel intensity per image (train sample)")
plt.xlabel("Mean intensity (0 = black, 255 = white)")
plt.ylabel("Number of images")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_intensity_hist.png", dpi=150)
plt.show()

display(train_sample.groupby("class")["mean_intensity"].describe().round(2))

If the two histograms overlap heavily, brightness alone cannot separate the classes →
the model must learn **shape and texture** patterns, which is exactly what a CNN is good at.

## 9. Average image per class

In [ ]:
def average_image(cls, n=300, size=(300, 300)):
    paths = df[(df["split"] == "train") & (df["class"] == cls)]["path"].sample(n, random_state=42)
    acc = np.zeros(size[::-1], dtype=np.float64)
    for p in paths:
        with Image.open(p) as im:
            acc += np.asarray(im.convert("L").resize(size), dtype=np.float64)
    return acc / n

avg_ok = average_image("ok")
avg_def = average_image("defective")
diff = np.abs(avg_def - avg_ok)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(avg_ok, cmap="gray");   axes[0].set_title("Average OK")
axes[1].imshow(avg_def, cmap="gray");  axes[1].set_title("Average defective")
im = axes[2].imshow(diff, cmap="hot"); axes[2].set_title("|Difference|")
for ax in axes:
    ax.axis("off")
fig.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_average_images.png", dpi=150)
plt.show()

Bright areas in the difference map show where defective and OK products differ **on average**.
Later, we can compare this with the model's Grad-CAM heatmaps.

## 10. Data leakage check: duplicate images between train and test
If the same image appears in both train and test, the test score will be **unrealistically high**.
We compare file hashes (MD5) to find exact duplicates.

In [ ]:
def file_md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df["md5"] = df["path"].apply(file_md5)

dup_within = df.duplicated("md5", keep=False).sum()
train_hashes = set(df[df["split"] == "train"]["md5"])
test_hashes = set(df[df["split"] == "test"]["md5"])
leak = train_hashes & test_hashes

print("Images that have an exact duplicate somewhere:", dup_within)
print("Exact duplicates shared by train and test:", len(leak))

> This only detects **exact** copies. The dataset is described as *augmented*, so near-duplicates
> (e.g. rotated versions of the same part) may still exist. Keep this in mind when interpreting
> very high test accuracy.

## 11. Summary & decisions for modeling
*(Fill this in after running the notebook. This summary goes into the README.)*

| Finding | Decision for the next notebook |
|---|---|
| Class balance: … | Use class weights / track defective recall |
| Image size & mode: … | Resize to 224×224, convert grayscale → 3-channel |
| Intensity overlap: … | Rely on CNN features, not simple thresholds |
| Train/test duplicates: … | Note as a limitation / remove duplicates |

**Next:** `02_training.ipynb` — baseline CNN, then transfer learning (MobileNetV2, EfficientNet-B0, ResNet50).